# Complete Cell-by-Cell Brain Tumor CNN Pipeline

This pipeline is modularized into distinct, run-ready cells:
1. **Environment & Paths Setup**
2. **Medical Data Augmentations & Generators**
3. **CNN Deep Architecture Construction**
4. **Optimization Callbacks & Training Loop**
5. **Hold-out Test Dataset Evaluation & Diagnostics**
6. **Single-Slice Inference Engine**


In [ ]:

# CELL 1: Imports and Directory Configuration
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Configure Paths matching your Kaggle directory
DATASET_PATH = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
TRAIN_DIR = os.path.join(DATASET_PATH, "Training")
TEST_DIR = os.path.join(DATASET_PATH, "Testing")
# Diagnostic Image Hyperparameters
IMG_SIZE = 128
BATCH_SIZE = 32

print("Core libraries loaded. Base paths and hyper-parameters configured successfully.")

import os
print(os.listdir("/root/.cache/kagglehub/datasets"))
# print(f"I gotCha")

```markdown
## Step 1: Data Augmentation & Generators
We introduce safe, medically realistic transformations (rotation, shifting, zooms, horizontal flips) only to the training set to prevent overfitting. The validation set is kept clean (only normalized).
```

# CELL 2: Data Generators & Class Mapping
# Augmentation settings to generalize across scanner variations
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# Only normalization for the validation split to preserve clinical integrity
val_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2
)

train_generator_cnn = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    subset="training",
    shuffle=True,
    seed=42
)

validation_generator_cnn = val_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    subset="validation",
    shuffle=False,
    seed=42
)

cnn_class_names = [k for k, v in sorted(train_generator_cnn.class_indices.items(), key=lambda item: item[1])]
print("\nMapped Diagnostic Classes:", cnn_class_names)

```markdown
## Step 2: CNN Architecture
We construct a deep network using convolutional layers (`Conv2D`), pooling layers (`MaxPooling2D`), and regularizers like `BatchNormalization` and `Dropout` to stabilize training.
```

# CELL 3: CNN Model Architecture Design
cnn_model = Sequential([
    # Block 1: Low-level edges and textures
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    # Block 2: Middle-level geometric tumor boundaries
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Block 3: High-level complex multi-layer configurations
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Dense Classifier
    Flatten(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

```markdown
## Step 3: Callbacks & Model Training
We use `EarlyStopping` to stop when validation loss stops improving, saving patient validation metrics and preventing model deterioration.
```

# CELL 4: Training with Dynamic Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_brain_tumor_cnn.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
]

print("Starting training loop... Please let this run completely without interruption.")
history_cnn = cnn_model.fit(
    train_generator_cnn,
    epochs=20,
    validation_data=validation_generator_cnn,
    callbacks=callbacks
)

```markdown
## Step 3b: Robust Training Execution
Let's run a complete, uninterrupted training cycle. We'll use a dynamic `ReduceLROnPlateau` scheduler. This acts like a fine-adjustment knob on a microscope, slowing down learning steps as the model nears optimal weight configurations to prevent it from overshooting stable features.
```

# CELL 5: Validation Diagnostics and Performance Curve
plt.figure(figsize=(14, 5))

# Accuracy Curves
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--')

# Loss Curves
plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history_cnn.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Training and Validation Loss', fontsize=12)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--')

plt.tight_layout()
plt.show()

```markdown
## Step 4: Metric Evaluation
We visually analyze the loss and accuracy slopes to ensure the model is converging healthily.
```

# CELL 6: Independent Hold-out Dataset Evaluation
from tensorflow.keras.models import load_model

# Ensure we analyze utilizing the absolute best saved iteration weight settings
eval_model = load_model('best_brain_tumor_cnn.keras')
test_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False
)

# Test evaluation loss/accuracy
test_loss, test_acc = eval_model.evaluate(test_generator)
print(f"\nGeneralization Accuracy: {test_acc*100:.2f}%")
print(f"Generalization Loss: {test_loss:.4f}")

# Precision/Recall Report
Y_pred = eval_model.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)

print("\n--- Clinical Classification Report ---")
print(classification_report(test_generator.classes, y_pred, target_names=cnn_class_names))

# Diagnostic Confusion Matrix
cm = confusion_matrix(test_generator.classes, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=cnn_class_names, yticklabels=cnn_class_names)
plt.ylabel('True Class Pathologies')
plt.xlabel('Predicted Diagnosis')
plt.title('Clinical Confusion Matrix')
plt.show()

```markdown
## Step 5: Clinical Prediction
Let's test this CNN against the query MRI image slice (`'glioma1.png'`).
```

# CELL 7: Single MRI Slice Inference Engine
from tensorflow.keras.preprocessing import image

def run_single_inference(img_filepath):
    # 1. Load, rescale, & scale dimensions to fit the model shape
    img_raw = image.load_img(img_filepath, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = image.img_to_array(img_raw) / 255.0
    img_arr = np.expand_dims(img_arr, axis=0)

    # 2. Perform feed-forward prediction pass
    pred_probs = eval_model.predict(img_arr)
    pred_idx = np.argmax(pred_probs[0])
    pred_class = cnn_class_names[pred_idx]
    conf_score = pred_probs[0][pred_idx] * 100

    # 3. Render raw visual and predicted values
    plt.figure(figsize=(6, 6))
    plt.imshow(img_raw)
    plt.title(f"Model Diagnostic Decision: {pred_class.upper()}\nConfidence: {conf_score:.2f}%", fontsize=12)
    plt.axis("off")
    plt.show()

    print("Diagnostic Class Probabilities:")
    for name, prob in zip(cnn_class_names, pred_probs[0]):
        print(f" - {name}: {prob*100:.2f}%")

# Change to any target local test file
test_slice_path = "menin2.jpeg"
if os.path.exists(test_slice_path):
    run_single_inference(test_slice_path)
else:
    print(f"File path not found: {test_slice_path}. Update the filepath above to test.")